In [99]:
from pathlib import Path
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

%load_ext autoreload
%autoreload 2
%matplotlib inline

cwd = Path.cwd().resolve()

# Locate helper_function by searching upward first, then common workspace locations.
def find_helper_folder(start: Path) -> Path:
    # 1) Directly in current/parent tree
    for p in [start, *start.parents]:
        candidate = p / "helper_function"
        if candidate.is_dir():
            return candidate

    raise FileNotFoundError("Could not find helper_function folder")


helper_path = find_helper_folder(cwd)
project_root = helper_path.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(helper_path) not in sys.path:
    sys.path.insert(0, str(helper_path))

print("Project root:", project_root)
print("Current working directory:", cwd)
print("helper_function path:", helper_path)
print("helper_function exists:", helper_path.exists())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project root: /Users/JackX/projects/trap_sim_nullspace
Current working directory: /Users/JackX/projects/trap_sim_nullspace/lionix_gate/output_file/dc
helper_function path: /Users/JackX/projects/trap_sim_nullspace/helper_function
helper_function exists: True


In [100]:
charge = 1.602176634e-19  # Coulombs
mass = 40 * 1.66053906660e-27  # kg (mass of Ca+ ion)
Omega = 2 * np.pi * 50e6  # rad/s (RF drive frequency)
pseudo_factor = charge / (4 * mass * Omega**2)

## Get grad and hessian of potential for each electrodes at x0, y0, z0

In [101]:
from post_processing import read_post_processed
from field_analysis import analyze_dc_pot

electrode_num = 27
rf_electrode_index = [13]
gnd_electrode_index = [1, 2, 14, 20, 26, 27]
excitation_electrodes = set(range(1, electrode_num + 1)) - set(gnd_electrode_index) - set(rf_electrode_index)
print("DC Excitation Electrodes:", excitation_electrodes)
excitation_electrodes = sorted(excitation_electrodes)
x, y, z, potentials = read_post_processed(filename="dc_sim_out.h5", electrode_names=list(excitation_electrodes))

x_axis = x[:, 0, 0]
y_axis = y[0, :, 0]
z_axis = z[0, 0, :]

DC Excitation Electrodes: {3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25}


In [102]:
n_points = 10

x0 = 0
y0 = 0
z0 = 49.99e-6

x0_index = np.argmin(np.abs(x_axis - x0))
y0_index = np.argmin(np.abs(y_axis - y0))
z0_index = np.argmin(np.abs(z_axis - z0))

# Index window around each center
half = max(0, n_points // 2)
ix_start = max(0, x0_index - half)
ix_end = min(x_axis.shape[0], x0_index + half + 1)
iy_start = max(0, y0_index - half)
iy_end = min(y_axis.shape[0], y0_index + half + 1)
iz_start = max(0, z0_index - half)
iz_end = min(z_axis.shape[0], z0_index + half + 1)

ix = np.arange(ix_start, ix_end)
iy = np.arange(iy_start, iy_end)
iz = np.arange(iz_start, iz_end)

# Windowed yz grid and pseudo-potential
X_abs = x[np.ix_(ix, iy, iz)]
Y_abs = y[np.ix_(ix, iy, iz)]
Z_abs = z[np.ix_(ix, iy, iz)]

dc_fields = {}


for i in excitation_electrodes:
    potential = potentials[i][np.ix_(ix, iy, iz)]
    
    coeff, phi, grad, hessian = analyze_dc_pot(X_abs, Y_abs, Z_abs, potential, x0, y0, z0)

    dc_fields[i] = (coeff, phi, grad, hessian)

To achieve axial trapping of ion at x0, y0, and z0 with $\omega_0 = 2 \pi\times1$ MHz, we would have grad = [0, 0, 0], hessian = $\alpha$ [[1, 0, 0], [0, -1/2, 0], [0, 0, -1/2]], with $\alpha = \frac{m \omega_0^2}{q}$

In [197]:
from field_analysis import fit_prep, print_field

omega = 2 * np.pi * 1e6

A, b = fit_prep(dc_fields, excitation_electrodes, target_phi = np.array([0]), target_grad = np.array([0,0,0]), target_hessian = np.array([[1, 0, 0], [0, -1/2, 0], [0, 0, -1/2]])*mass*omega**2/(charge))

In [198]:
x, *_ = np.linalg.lstsq(A, b, rcond=None)

In [200]:
for i in range(len(x)):
    print(excitation_electrodes[i], x[i])
print_field(A@x, delta_pos=True)

3 -1.9542851779990404
4 -2.588053160758451
5 -2.369291607631052
6 3.404101105195028
7 -1.7470872918927929
8 3.3977652207596494
9 -2.365004439695693
10 -2.562946713965834
11 -1.9238162331394653
12 -0.006926170655464465
15 -0.00684498889262973
16 -1.9560205594943203
17 -2.5862066456631743
18 -2.372195056359004
19 3.4009554815114025
21 -1.7427154136935616
22 3.389652220127416
23 -2.3586382053999975
24 -2.5721631424405387
25 -1.916053709284655
Phi: 1.4381379893381452e-09
Grad: [-9.60380664e-11  5.74495971e-08 -2.58451038e-11]
Hessian: [[ 1.63665986e+07  1.16051524e-09 -1.18780008e-09]
 [ 1.16051524e-09 -8.18329931e+06 -7.45058060e-09]
 [-1.18780008e-09 -7.45058060e-09 -8.18329931e+06]]
Delta Pos (um): [ 5.86793069e-12  7.02034655e-09 -3.15827429e-12]


Now to compare with voltage sets from before, using 5 electrodes from each side : 5, 6, 7, 8, 9, 18, 19, 21, 22, 23 and 12, 15 at the center

In [201]:
selected_electrodes = [5, 6, 7, 8, 9, 12, 15, 18, 19, 21, 22, 23]

In [202]:
omega = 2 * np.pi * 1e6
A, b = fit_prep(dc_fields, selected_electrodes, target_phi = None, target_grad = np.array([0,0,0]), target_hessian = np.array([[1, 0, 0], [0, -1/2, 0], [0, 0, -1/2]])*mass*omega**2/(charge))
x, *_ = np.linalg.lstsq(A, b, rcond=1e-10)

In [205]:
for i in range(len(x)):
    print(selected_electrodes[i], x[i])
print_field(A@x, delta_pos=True)

5 0.2896169383067789
6 0.5415987706912189
7 -2.1551389198015505
8 0.5400520649120429
9 0.30005116594264164
12 -0.2693633414148428
15 -0.26982053716776344
18 0.2898433897165037
19 0.5419318248370676
21 -2.1553869639249386
22 0.5387789632782392
23 0.2998907518035638
Grad: [ 1.14604215e-10 -2.51020538e-10 -9.27026012e-10]
Hessian: [[ 1.63665986e+07 -1.02521881e-07  9.87711246e-10]
 [-1.02521881e-07 -8.18329931e+06  7.45058060e-09]
 [ 9.87711246e-10  7.45058060e-09 -8.18329931e+06]]
Delta Pos (um): [-7.00232331e-12 -3.06747350e-11 -1.13282672e-10]


Compare to voltage sets we've been using.

In [206]:
x_anstaz = np.array([5.006, -2.408, -2.408, -2.408, 5.006, -0.604, -0.604, 5.006, -2.408, -2.408, -2.408, 5.006])
print_field(A @ x_anstaz, delta_pos=True)

Grad: [ -0.49604095   2.29139475 416.86133699]
Hessian: [[ 1.69271772e+07 -3.15080435e+03 -1.52258670e+04]
 [-3.15080435e+03  8.12344253e+06 -7.28496891e+04]
 [-1.52258670e+04 -7.28496891e+04 -2.50506197e+07]]
Delta Pos (um): [ 0.04424825 -0.13281992 16.6411188 ]


In [207]:
x_anstaz = np.array([0, 0.58095, -1.6915,0.58095, 0, -0.24083, -0.24083, 0, 0.58095, -1.6915,0.58095, 0])
print_field(A @ x_anstaz, delta_pos=True)

Grad: [ -0.35848744   0.7828535  214.24438414]
Hessian: [[ 1.29363356e+07  4.47644404e+03 -8.46184502e+03]
 [ 4.47644404e+03 -2.61246321e+04 -2.20522000e+04]
 [-8.46184502e+03 -2.20522000e+04 -1.29102110e+07]]
Delta Pos (um): [ 0.03301679 15.98677726 16.56762713]
